# Message Passing Neural Networks (MPNN) for Molecular Property Prediction

## A Comprehensive Tutorial

This notebook provides a complete guide to understanding and implementing Message Passing Neural Networks for predicting molecular properties using PyTorch Geometric.

**Based on:** *Neural Message Passing for Quantum Chemistry* by Gilmer et al. (2017)

---

## Table of Contents

1. [Introduction](#1.-Introduction)
2. [Understanding Graph Neural Networks](#2.-Understanding-Graph-Neural-Networks)
3. [MPNN Architecture](#3.-MPNN-Architecture)
4. [Data Preparation](#4.-Data-Preparation)
5. [Model Implementation](#5.-Model-Implementation)
6. [Training](#6.-Training)
7. [Evaluation](#7.-Evaluation)
8. [Visualization](#8.-Visualization)
9. [Exercises](#9.-Exercises)

---

## 1. Introduction

### What are Message Passing Neural Networks?

**Message Passing Neural Networks (MPNNs)** are a class of Graph Neural Networks designed to learn representations of graph-structured data. They are particularly powerful for molecular property prediction because molecules naturally form graphs:

- **Nodes** = Atoms (with features like atomic number, charge, etc.)
- **Edges** = Chemical bonds (with features like bond type, distance, etc.)

### Why Use MPNNs for Chemistry?

Traditional machine learning methods require manual feature engineering (molecular descriptors). MPNNs automatically learn relevant features from the molecular graph structure, making them:

✓ **End-to-end learnable** - No manual feature engineering
✓ **Expressive** - Can capture complex molecular patterns
✓ **Generalizable** - Work across different molecular properties
✓ **Interpretable** - Can understand which parts of molecules matter

### The QM9 Dataset

We'll use the **QM9 dataset**, which contains:
- 134,000 small organic molecules (up to 9 heavy atoms)
- 19 quantum chemical properties per molecule
- Properties include: HOMO/LUMO energies, dipole moment, polarizability, etc.

## 2. Understanding Graph Neural Networks

### The Core Idea

Graph Neural Networks work through **iterative message passing**:

```
1. Initialize: Each node starts with its feature vector
2. Message: Nodes send messages to their neighbors
3. Aggregate: Each node collects messages from neighbors
4. Update: Each node updates its representation
5. Repeat: Steps 2-4 for multiple iterations
6. Readout: Aggregate all node representations to graph level
```

### Intuition

Think of it like a social network:
- Each person (node) has information about themselves
- People talk to their friends (message passing)
- After several rounds of conversation, everyone knows about their extended network
- The number of iterations determines how far information can travel

For molecules:
- Each atom learns about its neighbors
- With 3 iterations, an atom can "see" atoms up to 3 bonds away
- This captures local chemical environments

## 3. MPNN Architecture

### Three Key Components

#### 3.1 Message Function

Computes messages from source nodes to target nodes:

$$m_{ij} = \text{Message}(h_i, h_j, e_{ij})$$

Where:
- $h_i$ = features of source node (atom)
- $h_j$ = features of target node (neighbor atom)
- $e_{ij}$ = features of edge (chemical bond)
- $m_{ij}$ = message sent from node i to node j

**Implementation:** Neural network that processes edge features and source node features.

#### 3.2 Update Function

Updates node representations based on aggregated messages:

$$h_i^{t+1} = \text{Update}(h_i^t, \sum_{j \in N(i)} m_{ij})$$

Where:
- $h_i^t$ = node representation at iteration t
- $N(i)$ = neighbors of node i
- $\sum$ = aggregation (sum, mean, or max)

**Implementation:** GRU (Gated Recurrent Unit) that combines old state with new messages.

#### 3.3 Readout Function

Aggregates node representations to graph-level prediction:

$$y = \text{Readout}(\{h_i^T | i \in G\})$$

Where:
- $h_i^T$ = final node representations from all layers
- $G$ = all nodes in the graph
- $y$ = predicted property

**Implementation:** Pool node features, then apply MLP for final prediction.

## 4. Data Preparation

Let's start by importing libraries and loading the data.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install torch torchvision
# !pip install torch-geometric
# !pip install matplotlib seaborn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.datasets import QM9
from torch.utils.data import Subset

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### Load QM9 Dataset

We'll predict the **HOMO (Highest Occupied Molecular Orbital) energy**, which is important for understanding molecular reactivity.

In [ ]:
from data_utils import (
    load_qm9_pyg, 
    QM9Properties, 
    get_qm9_statistics,
    create_data_splits,
    print_data_example
)

# Show available properties
print("Available QM9 Properties:")
print("=" * 80)
QM9Properties.print_all_properties()

# Load dataset for HOMO energy (index 2)
print("\n" + "=" * 80)
print("Loading QM9 dataset for HOMO energy prediction...")
dataset = load_qm9_pyg(root='../data/qm9_pyg', target_property='homo')
print(f"✓ Loaded {len(dataset)} molecules")

### Explore the Data

Let's look at what a single molecule looks like in our dataset.

In [ ]:
# Get a sample molecule
sample = dataset[0]
print_data_example(sample)

# Visualize molecule sizes
print("\nAnalyzing molecule sizes...")
num_nodes_list = [dataset[i].num_nodes for i in range(min(1000, len(dataset)))]
num_edges_list = [dataset[i].num_edges for i in range(min(1000, len(dataset)))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(num_nodes_list, bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Atoms')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Molecule Sizes')
axes[0].grid(True, alpha=0.3)

axes[1].hist(num_edges_list, bins=20, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Number of Bonds')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Bond Counts')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Average atoms per molecule: {np.mean(num_nodes_list):.2f}")
print(f"Average bonds per molecule: {np.mean(num_edges_list):.2f}")

### Create Train/Val/Test Splits

We'll use the standard QM9 split:
- **Training:** 100,000 molecules
- **Validation:** 10,000 molecules
- **Test:** Remaining molecules

In [ ]:
# Create splits
train_dataset, val_dataset, test_dataset = create_data_splits(dataset)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Compute normalization statistics (from training set only!)
print("\nComputing normalization statistics...")
mean, std = get_qm9_statistics(dataset, target_idx=0)
print(f"HOMO Energy - Mean: {mean:.4f} eV, Std: {std:.4f} eV")

# Visualize target distribution
targets = [dataset[i].y.item() for i in range(min(10000, len(train_dataset)))]
plt.figure(figsize=(10, 5))
plt.hist(targets, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(mean, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean:.2f}')
plt.xlabel('HOMO Energy (eV)')
plt.ylabel('Frequency')
plt.title('Distribution of HOMO Energy in Training Set')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Create DataLoaders

PyTorch Geometric's DataLoader automatically batches graphs of different sizes.

In [ ]:
# Create dataloaders
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# Inspect a batch
print("\nSample batch:")
for batch in train_loader:
    print(f"  Batch contains {batch.num_graphs} molecules")
    print(f"  Total nodes: {batch.num_nodes}")
    print(f"  Total edges: {batch.num_edges}")
    print(f"  Node features shape: {batch.x.shape}")
    print(f"  Edge features shape: {batch.edge_attr.shape}")
    print(f"  Targets shape: {batch.y.shape}")
    break

## 5. Model Implementation

Now let's load our clean MPNN implementation and understand each component.

In [ ]:
from mpnn_model import MPNN, create_mpnn_model

# Get feature dimensions from data
sample_batch = next(iter(train_loader))
node_dim = sample_batch.x.shape[1]
edge_dim = sample_batch.edge_attr.shape[1]
output_dim = 1  # Predicting a single property

print(f"Node feature dimension: {node_dim}")
print(f"Edge feature dimension: {edge_dim}")
print(f"Output dimension: {output_dim}")

# Create model
model = create_mpnn_model(
    node_dim=node_dim,
    edge_dim=edge_dim,
    output_dim=output_dim,
    hidden_dim=64,
    num_layers=3
).to(device)

print(f"\nModel: {model}")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

### Understanding Model Components

Let's break down what each part of the model does:

In [ ]:
# Test forward pass with a single batch
model.eval()
with torch.no_grad():
    sample_batch = sample_batch.to(device)
    output = model(sample_batch)
    print(f"Input: {sample_batch.num_graphs} molecules")
    print(f"Output shape: {output.shape}")
    print(f"Sample predictions: {output[:5].squeeze()}")

# Visualize model architecture
print("\n" + "="*80)
print("MODEL ARCHITECTURE")
print("="*80)
for name, module in model.named_children():
    print(f"\n{name.upper()}:")
    print(f"  {module}")

## 6. Training

Now we'll train the model to predict HOMO energies.

In [ ]:
# Training configuration
num_epochs = 50
learning_rate = 1e-3
patience = 10  # Early stopping patience

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

print(f"Training configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  Optimizer: Adam")
print(f"  Loss: MSE")
print(f"  Device: {device}")

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for batch in tqdm(loader, desc='Training', leave=False):
        batch = batch.to(device)
        
        # Normalize targets
        targets = (batch.y - mean) / std
        
        # Forward pass
        optimizer.zero_grad()
        output = model(batch)
        
        # Compute loss
        loss = criterion(output, targets)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
    
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating', leave=False):
            batch = batch.to(device)
            
            # Normalize targets
            targets = (batch.y - mean) / std
            
            # Forward pass
            output = model(batch)
            
            # Compute loss
            loss = criterion(output, targets)
            total_loss += loss.item() * batch.num_graphs
            
            # Denormalize for MAE calculation
            pred_denorm = output * std + mean
            target_denorm = batch.y
            
            all_predictions.append(pred_denorm.cpu())
            all_targets.append(target_denorm.cpu())
    
    # Calculate metrics
    all_predictions = torch.cat(all_predictions)
    all_targets = torch.cat(all_targets)
    mae = torch.mean(torch.abs(all_predictions - all_targets)).item()
    
    return total_loss / len(loader.dataset), mae

In [ ]:
# Training loop
print("Starting training...\n")

train_losses = []
val_losses = []
val_maes = []
best_val_loss = float('inf')
best_model_state = None
epochs_no_improve = 0

start_time = time.time()

for epoch in range(num_epochs):
    epoch_start = time.time()
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    train_losses.append(train_loss)
    
    # Validate
    val_loss, val_mae = evaluate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    val_maes.append(val_mae)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Check for improvement
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        epochs_no_improve = 0
        best_indicator = '🌟'
    else:
        epochs_no_improve += 1
        best_indicator = ''
    
    epoch_time = time.time() - epoch_start
    
    # Print progress
    print(f"Epoch {epoch+1:3d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val MAE: {val_mae:.4f} eV | "
          f"Time: {epoch_time:.1f}s {best_indicator}")
    
    # Early stopping
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping triggered after {epoch+1} epochs!")
        break

total_time = time.time() - start_time
print(f"\nTraining completed in {total_time/60:.2f} minutes")
print(f"Best validation loss: {best_val_loss:.4f}")

# Load best model
model.load_state_dict(best_model_state)
print("✓ Loaded best model checkpoint")

### Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot losses
epochs_range = range(1, len(train_losses) + 1)
axes[0].plot(epochs_range, train_losses, 'b-', label='Training Loss', linewidth=2)
axes[0].plot(epochs_range, val_losses, 'r-', label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss (normalized)')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot MAE
axes[1].plot(epochs_range, val_maes, 'g-', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (eV)')
axes[1].set_title('Validation Mean Absolute Error')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final validation MAE: {val_maes[-1]:.4f} eV")

## 7. Evaluation

Let's evaluate the trained model on the test set.

In [ ]:
# Evaluate on test set
print("Evaluating on test set...\n")
test_loss, test_mae = evaluate(model, test_loader, criterion, device)

print(f"Test Results:")
print(f"  MSE Loss: {test_loss:.4f}")
print(f"  MAE: {test_mae:.4f} eV")
print(f"  RMSE: {np.sqrt(test_loss * std**2):.4f} eV")

### Detailed Error Analysis

In [ ]:
# Get predictions for test set
model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Getting predictions'):
        batch = batch.to(device)
        output = model(batch)
        
        # Denormalize
        pred_denorm = output * std + mean
        
        all_predictions.append(pred_denorm.cpu())
        all_targets.append(batch.y.cpu())

predictions = torch.cat(all_predictions).numpy()
targets = torch.cat(all_targets).numpy()
errors = np.abs(predictions - targets)

print(f"Error statistics:")
print(f"  Mean: {errors.mean():.4f} eV")
print(f"  Std: {errors.std():.4f} eV")
print(f"  Median: {np.median(errors):.4f} eV")
print(f"  90th percentile: {np.percentile(errors, 90):.4f} eV")
print(f"  95th percentile: {np.percentile(errors, 95):.4f} eV")
print(f"  Max: {errors.max():.4f} eV")

## 8. Visualization

Let's visualize the model's predictions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Prediction vs Ground Truth
axes[0, 0].scatter(targets, predictions, alpha=0.3, s=10)
min_val = min(targets.min(), predictions.min())
max_val = max(targets.max(), predictions.max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[0, 0].set_xlabel('Ground Truth HOMO Energy (eV)')
axes[0, 0].set_ylabel('Predicted HOMO Energy (eV)')
axes[0, 0].set_title('Predictions vs Ground Truth')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Error distribution
axes[0, 1].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(errors.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {errors.mean():.3f}')
axes[0, 1].set_xlabel('Absolute Error (eV)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Prediction Errors')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Error vs Ground Truth
axes[1, 0].scatter(targets, errors, alpha=0.3, s=10)
axes[1, 0].set_xlabel('Ground Truth HOMO Energy (eV)')
axes[1, 0].set_ylabel('Absolute Error (eV)')
axes[1, 0].set_title('Error vs Ground Truth')
axes[1, 0].grid(True, alpha=0.3)

# 4. Residuals
residuals = predictions.flatten() - targets.flatten()
axes[1, 1].scatter(targets, residuals, alpha=0.3, s=10)
axes[1, 1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Ground Truth HOMO Energy (eV)')
axes[1, 1].set_ylabel('Residual (eV)')
axes[1, 1].set_title('Residual Plot')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Example Predictions

In [ ]:
# Show some example predictions
print("Example Predictions (first 20 test molecules):\n")
print(f"{'Index':<8} {'Ground Truth':<15} {'Prediction':<15} {'Error':<10}")
print("-" * 50)

for i in range(20):
    gt = targets[i, 0]
    pred = predictions[i, 0]
    err = abs(pred - gt)
    print(f"{i:<8} {gt:<15.4f} {pred:<15.4f} {err:<10.4f}")

# Find best and worst predictions
best_idx = errors.argmin()
worst_idx = errors.argmax()

print(f"\n🏆 Best prediction:")
print(f"   Index: {best_idx}")
print(f"   Ground truth: {targets[best_idx, 0]:.4f} eV")
print(f"   Prediction: {predictions[best_idx, 0]:.4f} eV")
print(f"   Error: {errors[best_idx, 0]:.4f} eV")

print(f"\n❌ Worst prediction:")
print(f"   Index: {worst_idx}")
print(f"   Ground truth: {targets[worst_idx, 0]:.4f} eV")
print(f"   Prediction: {predictions[worst_idx, 0]:.4f} eV")
print(f"   Error: {errors[worst_idx, 0]:.4f} eV")

### Save Model

In [ ]:
# Save trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'val_losses': val_losses,
    'val_maes': val_maes,
    'test_mae': test_mae,
    'mean': mean,
    'std': std,
    'config': {
        'node_dim': node_dim,
        'edge_dim': edge_dim,
        'hidden_dim': 64,
        'num_layers': 3,
        'output_dim': 1
    }
}, 'mpnn_qm9_homo.pth')

print("✓ Model saved to 'mpnn_qm9_homo.pth'")

## 9. Exercises

Try these exercises to deepen your understanding:

### Exercise 1: Different Target Property
Modify the code to predict a different property (e.g., LUMO energy, dipole moment). Compare the results.

### Exercise 2: Model Architecture
Experiment with different:
- Number of message passing layers (1-6)
- Hidden dimensions (32, 64, 128, 256)
- Aggregation methods ('add', 'mean', 'max')
- Pooling strategies in the readout

### Exercise 3: Multi-Task Learning
Modify the model to predict multiple properties simultaneously. How does this affect performance?

### Exercise 4: Attention Mechanism
Add attention mechanisms to the message passing or readout. Does it improve performance?

### Exercise 5: Transfer Learning
Train on one property, then fine-tune on another. Does pre-training help?

---

## Summary

In this tutorial, we:

1. ✓ Understood the concept of Message Passing Neural Networks
2. ✓ Loaded and explored the QM9 molecular dataset
3. ✓ Implemented a clean MPNN using PyTorch Geometric
4. ✓ Trained the model to predict HOMO energies
5. ✓ Evaluated performance and visualized results

### Key Takeaways:

- **MPNNs** are powerful for learning from graph-structured data
- **PyTorch Geometric** provides clean abstractions for graph neural networks
- **Message passing** allows atoms to share information with neighbors
- **Multiple layers** capture larger molecular contexts
- **Readout** aggregates node-level to graph-level representations

### Further Reading:

- Original Paper: [Neural Message Passing for Quantum Chemistry](https://arxiv.org/abs/1704.01212)
- PyTorch Geometric: [Documentation](https://pytorch-geometric.readthedocs.io/)
- QM9 Dataset: [Paper](https://www.nature.com/articles/sdata201422)

---

**Happy Learning! 🚀**